In [1]:
# Package installation omitted for local execution (dependencies already installed in local environment)
# !pip install -q pandas numpy scikit-learn statsmodels scipy


In [2]:
import pandas as pd

DATA_PATH = "data/happiscope_master.csv"

df = pd.read_csv(DATA_PATH)

print("Loaded local dataset:", df.shape)
df.head()


Loaded local dataset: (2363, 11)


,Country name,year,Life Ladder,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption,Positive affect,Negative affect
0,Afghanistan,2008,3.724,7.350,0.451,50.5,0.718,0.164,0.882,0.414,0.258
1,Afghanistan,2009,4.402,7.509,0.552,50.8,0.679,0.187,0.850,0.481,0.237
2,Afghanistan,2010,4.758,7.614,0.539,51.1,0.600,0.118,0.707,0.517,0.275
3,Afghanistan,2011,3.832,7.581,0.521,51.4,0.496,0.160,0.731,0.480,0.267
4,Afghanistan,2012,3.783,7.661,0.521,51.7,0.531,0.234,0.776,0.614,0.268


In [3]:
# Quick sanity check before modeling — this should look right before you trust anything below
print(df.shape)
print(df['year'].min(), '-', df['year'].max())
print(df['Country name'].nunique(), "unique countries")
print(df.isna().sum())

(2363, 11)
2005 - 2023
165 unique countries
Country name                          0
year                                  0
Life Ladder                           0
Log GDP per capita                   28
Social support                       13
Healthy life expectancy at birth     63
Freedom to make life choices         36
Generosity                           81
Perceptions of corruption           125
Positive affect                      24
Negative affect                      16
dtype: int64


In [4]:
import statsmodels.api as sm

FACTOR_COLS = ['Log GDP per capita', 'Social support', 'Healthy life expectancy at birth', 'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']

X = df[FACTOR_COLS]
X = sm.add_constant(X)
y = df['Life Ladder']

global_model = sm.OLS(y, X, missing='drop').fit()
print(global_model.summary())

                            OLS Regression Results                            
Dep. Variable:            Life Ladder   R-squared:                       0.753
Model:                            OLS   Adj. R-squared:                  0.753
Method:                 Least Squares   F-statistic:                     1066.
Date:                Mon, 28 Sep 2026   Prob (F-statistic):               0.00
Time:                        19:28:47   Log-Likelihood:                -1780.3
No. Observations:                2103   AIC:                             3575.
Df Residuals:                    2096   BIC:                             3614.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

In [5]:
# Pull the coefficients into a clean table you can screenshot / paste into the report
global_coefs = pd.DataFrame({
    'factor': global_model.params.index,
    'coefficient': global_model.params.values,
    'p_value': global_model.pvalues.values
})
print("R-squared:", round(global_model.rsquared, 3))
global_coefs.sort_values('coefficient', key=abs, ascending=False)

R-squared: 0.753


,factor,coefficient,p_value
2,Social support,2.627184,1.193793e-68
0,const,-1.992701,1.916898e-33
4,Freedom to make life choices,1.230838,9.114973e-28
6,Perceptions of corruption,-0.741727,7.915857e-20
5,Generosity,0.517391,4.632751e-10
1,Log GDP per capita,0.356598,8.693259e-55
3,Healthy life expectancy at birth,0.025835,1.417299e-15


In [6]:
# [OBSOLETE / DRAFT] Cell 5 is an earlier unconstrained draft. The authoritative yearly regression is Cell 6.
yearly_coefs = []

for yr in sorted(df['year'].unique()):
    yr_df = df[df['year'] == yr]
    if len(yr_df) < 20:   # skip years with too few countries to trust a regression
        continue
    X_yr = sm.add_constant(yr_df[FACTOR_COLS])
    y_yr = yr_df['Life Ladder']
    model_yr = sm.OLS(y_yr, X_yr, missing='drop').fit()

    # Check if r_squared is finite before adding to results
    if pd.isna(model_yr.rsquared) or not pd.api.types.is_number(model_yr.rsquared) or model_yr.rsquared == -float('inf'):
        print(f"Skipping year {yr} due to invalid R-squared: {model_yr.rsquared}")
        continue

    row = {'year': yr, 'r_squared': model_yr.rsquared}
    for factor in FACTOR_COLS:
        row[factor] = model_yr.params.get(factor, None)
    yearly_coefs.append(row)

yearly_df = pd.DataFrame(yearly_coefs).set_index('year')
yearly_df

Skipping year 2005 due to invalid R-squared: nan


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


,r_squared,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption
year,,,,,,,
2006,0.866236,0.353774,3.225608,0.016808,0.602487,1.188533,-1.129269
2007,0.734642,0.558975,0.748573,0.006393,1.140069,1.209415,-0.666522
2008,0.841607,0.467571,1.787043,0.013996,0.874537,0.797190,-1.005807
2009,0.672935,0.310917,1.848259,0.035356,1.051547,0.666688,-0.447218
2010,0.742960,0.398443,1.808980,0.034651,0.719518,1.309172,-1.012350
2011,0.726514,0.376491,2.484873,0.017427,0.726063,1.130696,-0.667999
2012,0.788935,0.425684,3.034552,0.019968,1.290297,0.690689,-0.619470
2013,0.730292,0.358947,2.386391,0.048388,1.462860,0.383084,-0.512502
2014,0.772471,0.273801,2.262130,0.059814,1.385976,0.396349,-0.659412


In [7]:
# Fix: Ensure each yearly regression uses at least 30 complete-case observations.
# This prevents unstable coefficients from years with sparse data.

corrected_yearly_coefs = []

for yr in sorted(df['year'].unique()):
    yr_df = df[df['year'] == yr]

    # Select relevant columns for complete-case check
    cols_to_check = FACTOR_COLS + ['Life Ladder']
    complete_cases_df = yr_df[cols_to_check].dropna()

    # Check for complete-case count before fitting OLS
    if len(complete_cases_df) < 30: # New threshold of at least 30 complete cases
        print(f"Skipping year {yr} due to insufficient complete cases ({len(complete_cases_df)} < 30).")
        continue

    X_yr_corrected = sm.add_constant(complete_cases_df[FACTOR_COLS])
    y_yr_corrected = complete_cases_df['Life Ladder']

    model_yr_corrected = sm.OLS(y_yr_corrected, X_yr_corrected).fit() # missing='drop' not needed here as we already dropped NaNs

    # Skip if R-squared is invalid (e.g., -inf, NaN)
    if pd.isna(model_yr_corrected.rsquared) or model_yr_corrected.rsquared == -float('inf'):
        print(f"Skipping year {yr} due to invalid R-squared: {model_yr_corrected.rsquared}")
        continue

    row = {'year': yr, 'r_squared': model_yr_corrected.rsquared}
    for factor in FACTOR_COLS:
        row[factor] = model_yr_corrected.params.get(factor, None)
    corrected_yearly_coefs.append(row)

# Rebuild yearly_df, earliest_year, latest_year with corrected data
corrected_yearly_df = pd.DataFrame(corrected_yearly_coefs).set_index('year')
corrected_earliest_year = corrected_yearly_df.index.min()
corrected_latest_year = corrected_yearly_df.index.max()

# Rebuild comparison table, dropping pct_change and sorting by absolute 'change'
corrected_comparison = pd.DataFrame({
    f'{corrected_earliest_year}': corrected_yearly_df.loc[corrected_earliest_year, FACTOR_COLS],
    f'{corrected_latest_year}': corrected_yearly_df.loc[corrected_latest_year, FACTOR_COLS],
})
corrected_comparison['change'] = corrected_comparison[f'{corrected_latest_year}'] - corrected_comparison[f'{corrected_earliest_year}']

# Display the corrected yearly_df and comparison
display(corrected_yearly_df)
display(corrected_comparison.sort_values('change', key=abs, ascending=False))

# Update global variables for downstream use
yearly_df = corrected_yearly_df
earliest_year = corrected_earliest_year
latest_year = corrected_latest_year
comparison = corrected_comparison.sort_values('change', key=abs, ascending=False)

Skipping year 2005 due to insufficient complete cases (1 < 30).


,r_squared,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption
year,,,,,,,
2006,0.866236,0.353774,3.225608,0.016808,0.602487,1.188533,-1.129269
2007,0.734642,0.558975,0.748573,0.006393,1.140069,1.209415,-0.666522
2008,0.841607,0.467571,1.787043,0.013996,0.874537,0.797190,-1.005807
2009,0.672935,0.310917,1.848259,0.035356,1.051547,0.666688,-0.447218
2010,0.742960,0.398443,1.808980,0.034651,0.719518,1.309172,-1.012350
2011,0.726514,0.376491,2.484873,0.017427,0.726063,1.130696,-0.667999
2012,0.788935,0.425684,3.034552,0.019968,1.290297,0.690689,-0.619470
2013,0.730292,0.358947,2.386391,0.048388,1.462860,0.383084,-0.512502
2014,0.772471,0.273801,2.262130,0.059814,1.385976,0.396349,-0.659412


,2006,2023,change
Freedom to make life choices,0.602487,2.080491,1.478004
Generosity,1.188533,-0.155115,-1.343648
Social support,3.225608,3.553997,0.328389
Log GDP per capita,0.353774,0.174366,-0.179408
Healthy life expectancy at birth,0.016808,0.027631,0.010823
Perceptions of corruption,-1.129269,-1.124113,0.005156


In [8]:
# [OBSOLETE / REDUNDANT] Cell 7 is redundant; the authoritative earliest vs latest comparison was computed in Cell 6.
# Directly compare the earliest vs. latest year you have data for
earliest_year = yearly_df.index.min()
latest_year = yearly_df.index.max()

comparison = pd.DataFrame({
    f'{earliest_year}': yearly_df.loc[earliest_year, FACTOR_COLS],
    f'{latest_year}': yearly_df.loc[latest_year, FACTOR_COLS],
})
comparison['change'] = comparison[f'{latest_year}'] - comparison[f'{earliest_year}']
comparison['pct_change'] = (comparison['change'] / comparison[f'{earliest_year}'].abs()) * 100
comparison.sort_values('change', key=abs, ascending=False)

,2006,2023,change,pct_change
Freedom to make life choices,0.602487,2.080491,1.478004,245.317018
Generosity,1.188533,-0.155115,-1.343648,-113.050930
Social support,3.225608,3.553997,0.328389,10.180698
Log GDP per capita,0.353774,0.174366,-0.179408,-50.712661
Healthy life expectancy at birth,0.016808,0.027631,0.010823,64.388475
Perceptions of corruption,-1.129269,-1.124113,0.005156,0.456576


In [9]:
reg = pd.read_csv("data/country_region.csv")

df["region"] = df["Country name"].map(
    reg.set_index("country")["region"]
)

region_col = "region"

print(df["region"].value_counts(dropna=False))


region
Asia             705
Europe           651
Africa           588
North America    204
South America    181
Oceania           34
Name: count, dtype: int64


In [10]:
region_coefs = []

for region, region_df in df.groupby(region_col):
    if len(region_df) < 20:
        continue
    X_r = sm.add_constant(region_df[FACTOR_COLS])
    y_r = region_df['Life Ladder']
    model_r = sm.OLS(y_r, X_r, missing='drop').fit()

    row = {'region': region, 'r_squared': model_r.rsquared, 'n_obs': len(region_df)}
    for factor in FACTOR_COLS:
        row[factor] = model_r.params.get(factor, None)
    region_coefs.append(row)

region_df_results = pd.DataFrame(region_coefs).set_index('region')
region_df_results

,r_squared,n_obs,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption
region,,,,,,,,
Africa,0.268806,588,0.275715,1.066051,0.016300,0.702743,0.701364,0.308867
Asia,0.606023,705,0.449714,2.775812,0.002717,0.720492,0.406338,-0.145651
Europe,0.822180,651,0.413205,2.458999,0.034235,1.204877,0.362287,-1.135822
North America,0.640453,204,0.415937,2.385370,0.022049,0.840625,-0.053497,-0.753841
Oceania,0.552472,34,0.321143,1.730411,-0.183236,1.808114,0.245065,0.250076
South America,0.472594,181,0.399936,2.641640,-0.016627,0.625413,-0.039373,-0.816270


In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

X_rf = df[FACTOR_COLS].dropna()
y_rf = df.loc[X_rf.index, 'Life Ladder']

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

test_r2 = rf.score(X_test, y_test)
importances = pd.Series(rf.feature_importances_, index=FACTOR_COLS).sort_values(ascending=False)

print("Random Forest test R-squared:", round(test_r2, 3))
print("\nFeature importances:")
print(importances)

Random Forest test R-squared: 0.866

Feature importances:
Log GDP per capita                  0.671580
Social support                      0.093497
Healthy life expectancy at birth    0.090504
Freedom to make life choices        0.068248
Perceptions of corruption           0.038761
Generosity                          0.037410
dtype: float64


In [12]:
# [OBSOLETE / BIASED COMPARISON] Cell 11 compares raw OLS vs MDI. The authoritative corrected comparison using standardized OLS and permutation importance is in Cells 12 and 13.
# Do the linear regression and the Random Forest agree on the ranking of factors?
ols_ranking = global_coefs[global_coefs['factor'] != 'const'].set_index('factor')['coefficient'].abs().sort_values(ascending=False)
rf_ranking = importances

comparison_ranks = pd.DataFrame({
    'ols_rank': ols_ranking.rank(ascending=False),
    'rf_rank': rf_ranking.rank(ascending=False)
})
correlation, p_value = spearmanr(comparison_ranks['ols_rank'], comparison_ranks['rf_rank'])

print(comparison_ranks)
print(f"\nSpearman rank correlation between OLS and Random Forest: {correlation:.2f} (p={p_value:.3f})")
print("A high positive correlation here is a good sentence for your report:")
print("'A linear and non-linear model independently agree on which factors matter most.'")

                                  ols_rank  rf_rank
Freedom to make life choices           2.0      4.0
Generosity                             4.0      6.0
Healthy life expectancy at birth       6.0      3.0
Log GDP per capita                     5.0      1.0
Perceptions of corruption              3.0      5.0
Social support                         1.0      2.0

Spearman rank correlation between OLS and Random Forest: -0.09 (p=0.872)
A high positive correlation here is a good sentence for your report:
'A linear and non-linear model independently agree on which factors matter most.'


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

# 1. Standardize predictors and refit OLS
# First, drop NaNs from X and y to ensure consistent indices for StandardScaler
X_complete = df[FACTOR_COLS].dropna()
y_complete = df.loc[X_complete.index, 'Life Ladder']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_complete)
X_scaled_df = pd.DataFrame(X_scaled, columns=FACTOR_COLS, index=X_complete.index)
X_scaled_df = sm.add_constant(X_scaled_df) # Add constant to scaled data

scaled_ols_model = sm.OLS(y_complete, X_scaled_df).fit()

# Standardized coefficients (excluding constant)
standardized_ols_coefs = scaled_ols_model.params.drop('const').abs().sort_values(ascending=False)

print("Standardized OLS Coefficients (absolute value, sorted):")
display(standardized_ols_coefs)

# 2. Compute permutation importance for Random Forest
# Use X_test, y_test from the original RF setup
perm_importance_results = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
permutation_importances_series = pd.Series(perm_importance_results.importances_mean, index=X_test.columns).sort_values(ascending=False)

print("\nRandom Forest Permutation Importances (mean, sorted):")
display(permutation_importances_series)

Standardized OLS Coefficients (absolute value, sorted):


Log GDP per capita                  0.411417
Social support                      0.324806
Healthy life expectancy at birth    0.180673
Freedom to make life choices        0.170676
Perceptions of corruption           0.136196
Generosity                          0.083852
dtype: float64


Random Forest Permutation Importances (mean, sorted):


Log GDP per capita                  0.630272
Healthy life expectancy at birth    0.138878
Social support                      0.110152
Freedom to make life choices        0.086596
Generosity                          0.037574
Perceptions of corruption           0.035754
dtype: float64

In [14]:
# 3. Redo Spearman rank correlation

# Original (biased) comparison (if Cell 11 was executed)
if "comparison_ranks" in globals():
    print("\n--- Original (Biased) Comparison ---")
    display(comparison_ranks)
    old_correlation, old_p_value = spearmanr(comparison_ranks["ols_rank"], comparison_ranks["rf_rank"])
    print(f"Spearman rank correlation (Original): {old_correlation:.2f} (p={old_p_value:.3f})")

# Corrected comparison (using standardized OLS coefs and permutation importances)
corrected_ols_ranking = standardized_ols_coefs.rank(ascending=False)
corrected_rf_ranking = permutation_importances_series.rank(ascending=False)

corrected_comparison_ranks = pd.DataFrame({
    "ols_rank_std": corrected_ols_ranking,
    "rf_rank_perm": corrected_rf_ranking
})

print("\n--- Corrected Comparison ---")
display(corrected_comparison_ranks)

new_correlation, new_p_value = spearmanr(corrected_comparison_ranks["ols_rank_std"], corrected_comparison_ranks["rf_rank_perm"])
print(f"Spearman rank correlation (Corrected): {new_correlation:.2f} (p={new_p_value:.3f})")



--- Original (Biased) Comparison ---


,ols_rank,rf_rank
Freedom to make life choices,2.0,4.0
Generosity,4.0,6.0
Healthy life expectancy at birth,6.0,3.0
Log GDP per capita,5.0,1.0
Perceptions of corruption,3.0,5.0
Social support,1.0,2.0


Spearman rank correlation (Original): -0.09 (p=0.872)

--- Corrected Comparison ---


,ols_rank_std,rf_rank_perm
Freedom to make life choices,4.0,4.0
Generosity,6.0,5.0
Healthy life expectancy at birth,3.0,2.0
Log GDP per capita,1.0,1.0
Perceptions of corruption,5.0,6.0
Social support,2.0,3.0


Spearman rank correlation (Corrected): 0.89 (p=0.019)


In [15]:
import os

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

global_coefs.to_csv(f"{OUT_DIR}/global_regression_coefficients.csv", index=False)
yearly_df.to_csv(f"{OUT_DIR}/regression_coefficients_by_year.csv")
comparison.to_csv(f"{OUT_DIR}/factor_change_{earliest_year}_vs_{latest_year}.csv")
region_df_results.to_csv(f"{OUT_DIR}/regression_coefficients_by_region.csv")
importances.to_csv(f"{OUT_DIR}/random_forest_feature_importances.csv")

# Consolidated comparison table for Streamlit application
importance_comparison_df = pd.DataFrame({
    "factor": FACTOR_COLS,
    "standardized_ols": [scaled_ols_model.params[f] for f in FACTOR_COLS],
    "rf_permutation": [permutation_importances_series[f] for f in FACTOR_COLS],
    "rf_test_r2": test_r2
})

# Verify schema and update data/importance_comparison.csv
expected_cols = ["factor", "standardized_ols", "rf_permutation", "rf_test_r2"]
if list(importance_comparison_df.columns) == expected_cols:
    importance_comparison_df.to_csv("data/importance_comparison.csv", index=False)
    print("Updated data/importance_comparison.csv with reproduced model results")
else:
    print("Schema mismatch! Existing data/importance_comparison.csv NOT updated.")

print("Saved all outputs to:", OUT_DIR)
print(os.listdir(OUT_DIR))


Updated data/importance_comparison.csv with reproduced model results
Saved all outputs to: outputs
['regression_coefficients_by_region.csv', 'global_regression_coefficients.csv', 'random_forest_feature_importances.csv', 'factor_change_2006_vs_2023.csv', 'regression_coefficients_by_year.csv']
